# Inspect preparation before docking

This example uses the public `assess_preparation` tool independently of Vina. It inspects declared preparation markers, not chemical validity. MolSysMT supplies the original molecular input; general chemical coverage remains its responsibility.

In [1]:
import sys
from pathlib import Path
import json
import dockingmt as dmt
import molsysmt as msm

assert sys.version_info[:2] == (3, 14)
assert 'molsyssuite@uibcdf_3.14' in sys.executable
print(sys.executable)
print('MolSysMT source:', Path(msm.__file__).resolve())

/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python
MolSysMT source: /home/diego/repos@uibcdf/dockingmt/.pytest_cache/codex-source-archives/dockingmt-template-provider-c19a47ada0c2279029abfa296cf915560610ad9a/molsysmt/__init__.py


Prepare the original six-atom 181L BNZ ligand through the existing MolSysMT input route, then inspect its declared preparation. No docking calculation occurs.

In [2]:
source = msm.systems['T4 lysozyme L99A']['181l.pdb']
ligand = dmt.prepare_ligand(source, selection="group_name=='BNZ'")
report = dmt.assess_preparation(ligand)
assert report['assessment'] == 'provisional'
assert report['provisional_reason_codes'] == ['zero_placeholder_charges', 'heuristic_atom_types']
print(json.dumps(report, indent=2))

{
  "schema_version": "1.0",
  "scope": "declared_preparation_metadata",
  "assessment": "provisional",
  "provisional_reason_codes": [
    "zero_placeholder_charges",
    "heuristic_atom_types"
  ],
  "provisional_reasons": [
    "zero-placeholder partial charges",
    "heuristic AutoDock atom types"
  ],
  "evidence": {
    "charge_source": "zero_placeholder",
    "atom_type_source": "element_group_heuristic"
  }
}


External paths stay unassessed. This declaration-only inspection does not open the file or validate its PDBQT content. Reports are detached and can be saved separately; editing one does not alter the prepared object.

In [3]:
external = dmt.assess_preparation(Path('externally-prepared.pdbqt'))
assert external['assessment'] == 'unassessed'
assert not external['provisional_reason_codes']
print(json.dumps(external, indent=2))
report['provisional_reason_codes'].clear()
assert dmt.assess_preparation(ligand)['provisional_reason_codes']
print('Fresh inspection preserves the original provisional evidence.')

{
  "schema_version": "1.0",
  "scope": "declared_preparation_metadata",
  "assessment": "unassessed",
  "provisional_reason_codes": [],
  "provisional_reasons": [],
  "evidence": {
    "charge_source": null,
    "atom_type_source": null
  }
}
Fresh inspection preserves the original provisional evidence.


Vina consumes this same public function. Known provisional input requires explicit exploratory opt-in; a completed chemical template transfer or successful engine run does not certify parameterization. See the [contract and guards](preparation_assessment.md).